# PostgreSQL SQL Interview Practice — 80 Questions

This notebook contains **10 questions for each of the 8 interview areas** from the supplied scope (**80 total**).

**Database:** PostgreSQL (configured via `db_config.ini`)  
**Database name:** `sql_interview`  
**Connection:** Configure `db_config.ini` once (see `README.md`).  
**Schema:** default `public`

First configure `db_config.ini`, then run `01_sql_interview_realistic_reseed_80_coverage.ipynb` once to create the practice tables. Then use the SQL code cell directly below each question to write and execute your answer.

> VS Code note: select the Python kernel from your project `venv`.


Each question includes the expected **tables and their columns** so you can focus on writing the SQL without repeatedly looking up the schema.


## Notebook setup

Run these cells once when you open the notebook.

In [ ]:
# Install packages used by this notebook into the active Jupyter kernel.
%pip install -q psycopg2-binary pandas

In [ ]:
import psycopg2
from IPython.display import display
import pandas as pd

# pandas is used only to display query results nicely.
# Install it if it is not already available in your venv.


In [ ]:
# Import the shared database configuration regardless of whether VS Code
# starts the kernel in coding-exercises/ or coding-exercises/sql/.
from pathlib import Path
import sys
_SQL_DIR = next((p for p in (Path.cwd(), Path.cwd() / "sql")
                 if (p / "db_connection.py").is_file()), None)
if _SQL_DIR is None:
    raise FileNotFoundError(
        "Cannot locate sql/db_connection.py. Open coding-exercises/ or its sql/ folder in VS Code."
    )
if str(_SQL_DIR) not in sys.path:
    sys.path.insert(0, str(_SQL_DIR))
from db_connection import connect, connection_settings

conn = connect(autocommit=True)
print(f"Connected to {connection_settings()['dbname']} as {connection_settings()['user']}")


In [ ]:
# Helper used by every answer cell.
# Write PostgreSQL in the triple-quoted string and run the cell.
def run_sql(sql):
    with conn.cursor() as cur:
        cur.execute(sql)

        # SELECT / WITH queries return rows. Other statements do not.
        if cur.description is None:
            print("Query executed successfully.")
            return None

        columns = [desc[0] for desc in cur.description]
        rows = cur.fetchall()

    df = pd.DataFrame(rows, columns=columns)
    display(df)
    #print(f"{len(df)} row(s)")
    #return df

# Connection check
run_sql("SELECT current_database() AS database, current_user AS user_name, current_schema() AS schema;")


## 1. Joins


### Q01 — Customer Order Summary

Return each customer who has placed at least one order, with customer_id, full_name, number_of_orders, and total_order_amount. Use INNER JOIN.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here

select
    c.customer_id,
    c.first_name || ' ' || c.last_name as full_name,
    count(o.order_id) as number_of_orders,
    sum(o.total_amount) as total_order_amount
from
    customers c
join orders o on c.customer_id = o.customer_id
group by c.customer_id
"""

run_sql(sql)


### Q02 — Customers Without Orders

Find customers who have never placed an order. Return customer_id, full_name, city, and signup_date. Use LEFT JOIN.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here
select
    c.customer_id,
    c.first_name || ' ' || c.last_name as full_name,
    c.city,
    c.signup_date
from
    customers c
left join orders o on c.customer_id = o.customer_id
where o.order_id is null


"""

run_sql(sql)


### Q03 — Orders With Customer and Employee

Return order_id, order_date, customer name, employee name, and total_amount for all orders.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title


In [ ]:
sql = """
-- Write your SQL answer here

select
    o.order_id,
    o.order_date,
    c.customer_id,
    c.first_name || ' ' || c.last_name as customer_name,
    e.first_name || ' ' || e.last_name as employee_name,
    o.total_amount
from
    orders o
join customers c on c.customer_id = o.customer_id
left join employees e on e.employee_id = o.sales_employee_id;

"""

run_sql(sql)


### Q04 — Employees and Managers

Using a SELF JOIN, return every employee with their manager's name. Keep employees who have no manager.  
**Tables:** \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title


In [ ]:
sql = """
-- Write your SQL answer here
select
    e1.first_name || ' ' || e1.last_name as employee_name,
    e2.first_name || ' ' || e2.last_name as manager_name
from
    employees e1
left join employees e2 on e1.manager_id = e2.employee_id

"""

run_sql(sql)


### Q05 — Products Never Ordered

Find products that have never appeared in order_items. Return product_id, product_name, category, and unit_price.  
**Tables:** \
**`products:`** product_id, sku, product_name, category, unit_price, stock_quantity, reorder_level, active \
**`order_items:`** order_item_id, order_id, product_id, quantity, unit_price


In [ ]:
sql = """
-- Write your SQL answer here
select
    p.product_id,
    p.product_name,
    p.category,
    p.unit_price
from
    products p
left join order_items oi on p.product_id = oi.product_id
where oi.order_item_id is null;

"""

run_sql(sql)


### Q06 — Customer Support Tickets

Return each support ticket with the customer name and the employee assigned to it. Keep unassigned tickets.  
**Tables:** \
**`support_tickets:`** ticket_id, customer_id, assigned_employee_id, created_at, resolved_at, priority, status, subject \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title


In [ ]:
sql = """
-- Write your SQL answer here
select
    st.ticket_id,
    c.first_name || ' ' || c.last_name as customer_name,
    e.first_name || ' ' || e.last_name as employee_name
from
    support_tickets st
join customers c on c.customer_id = st.customer_id
left join employees e on c.customer_id = e.employee_id

"""

run_sql(sql)


### Q07 — Order Line Details

Return order_id, product_name, quantity, unit_price at sale, and line_total for every order item.  
**Tables:** \
**`order_items:`** order_item_id, order_id, product_id, quantity, unit_price \
**`products:`** product_id, sku, product_name, category, unit_price, stock_quantity, reorder_level, active


In [ ]:
sql = """
-- Write your SQL answer here
select
    oi.order_id,
    p.product_name,
    oi.quantity,
    oi.unit_price,
    (oi.quantity * p.unit_price) as line_total
from
    order_items oi
join products p on p.product_id = oi.product_id


"""

run_sql(sql)


### Q08 — Same-City Customers

Use a SELF JOIN to find unique pairs of different customers who live in the same city. Do not return reversed duplicate pairs.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date


In [ ]:
sql = """
-- Write your SQL answer here
select
    c1.first_name || ' ' || c1.last_name as customer_1,
    c2.first_name || ' ' || c2.last_name as customer_2
from
    customers c1
join customers c2 on c1.city = c2.city and c1.customer_id < c2.customer_id

"""

run_sql(sql)


### Q09 — Projects and Team Members

Return each project and all employees assigned to it, including role_on_project and allocation_pct. Keep projects with no assignments.  
**Tables:** \
**`projects:`** project_id, project_name, start_date, end_date, status \
**`project_assignments:`** assignment_id, project_id, employee_id, role_on_project, allocation_pct \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title


In [ ]:
sql = """
-- Write your SQL answer here
select
    p.project_id,
    p.project_name,
    e.first_name || ' ' || e.last_name as employee_name,
    pa.role_on_project,
    pa.allocation_pct
from
    projects p
left join project_assignments pa on pa.project_id = p.project_id
join employees e on pa.employee_id = e.employee_id

"""

run_sql(sql)


### Q10 — Orders With Payment Status

Return every order with customer name and payment status. Keep orders that do not yet have a payment record.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`payments:`** payment_id, order_id, payment_date, amount, payment_method, payment_status


In [ ]:
sql = """
-- Write your SQL answer here
select
    o.order_id,
    c.first_name || ' ' || c.last_name as customer_name,
    p.payment_status
from
    orders o
join customers c on c.customer_id = o.customer_id
left join payments p on p.order_id = o.order_id;


"""

run_sql(sql)


## 2. Aggregations


### Q11 — Orders by Status

Count the number of orders in each order status.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here
select
    status,
    count(*) as order_count
from orders
group by status


"""

run_sql(sql)


### Q12 — Revenue by Customer

Calculate total completed-order revenue per customer and show only customers with more than $2,000 in completed revenue.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here
select
    c.customer_id,
    c.first_name || ' ' || c.last_name as customer_name,
    sum(o.total_amount) as completed_revenue
from customers c
join orders o on c.customer_id = o.customer_id
where o.status = 'Completed'
group by c.customer_id
having sum(o.total_amount) > 2000
order by completed_revenue DESC;

"""

run_sql(sql)


### Q13 — Department Salary Statistics

For each department, return employee_count, minimum salary, maximum salary, and average salary.  
**Tables:** \
**`departments:`** department_id, department_name, location \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q14 — Monthly Revenue

Calculate completed-order revenue by calendar month.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q15 — Conditional Order Counts

For each customer, count total orders, completed orders, cancelled orders, and pending orders using conditional aggregation.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q16 — Product Sales Performance

For each product, calculate units_sold, number_of_order_lines, and sales_revenue from order_items.  
**Tables:** \
**`products:`** product_id, sku, product_name, category, unit_price, stock_quantity, reorder_level, active \
**`order_items:`** order_item_id, order_id, product_id, quantity, unit_price


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q17 — High-Volume Cities

Return cities with at least 3 customers and show customer_count.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q18 — Ticket Status Summary

For each ticket priority, count total tickets, open tickets, resolved tickets, and closed tickets.  
**Tables:** \
**`support_tickets:`** ticket_id, customer_id, assigned_employee_id, created_at, resolved_at, priority, status, subject


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q19 — Average Order Value by Segment

For each customer segment, calculate order_count, total_revenue, and average_order_value.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q20 — Department Project Allocation

For each department, calculate the number of distinct employees assigned to projects and the average allocation percentage.  
**Tables:** \
**`departments:`** department_id, department_name, location \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title \
**`project_assignments:`** assignment_id, project_id, employee_id, role_on_project, allocation_pct


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


## 3. Window Functions


### Q21 — Latest Order per Customer

Return the latest order for each customer using ROW_NUMBER().  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q22 — Salary Rank by Department

Rank employees by salary within each department using RANK().  
**Tables:** \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q23 — Dense Salary Rank

Rank employees by salary within each department using DENSE_RANK() and explain from the output where it differs from RANK().  
**Tables:** \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q24 — Previous Customer Order

For every customer order, show the previous order date and previous order amount using LAG().  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q25 — Next Event Time

For each user in events, show each event and the next event timestamp using LEAD().  
**Tables:** \
**`events:`** event_id, user_id, event_type, event_ts, session_id, page


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q26 — Running Customer Spend

For each customer, show every order and a running total of order_amount ordered by date and order_id.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q27 — Department Salary vs Average

For every employee, show salary, department average salary, and the difference between salary and department average.  
**Tables:** \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q28 — Order Sequence Number

Number every customer's orders from earliest to latest using ROW_NUMBER().  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q29 — Running Product Units

For each product, show order dates and cumulative units sold over time.  
**Tables:** \
**`order_items:`** order_item_id, order_id, product_id, quantity, unit_price \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q30 — Customer Order Gap

For each customer order, calculate the number of days since the customer's previous order.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


## 4. CTEs & Subqueries


### Q31 — Above-Average Salary

Find employees whose salary is above the company-wide average salary using a subquery.  
**Tables:** \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q32 — Above Department Average

Using a CTE, find employees earning more than their own department's average salary.  
**Tables:** \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q33 — Customers Above Average Spend

Find customers whose total order spend is greater than the average total spend across customers.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q34 — Best-Selling Products

Using one or more CTEs, return products whose units sold are above the average units sold per product.  
**Tables:** \
**`products:`** product_id, sku, product_name, category, unit_price, stock_quantity, reorder_level, active \
**`order_items:`** order_item_id, order_id, product_id, quantity, unit_price


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q35 — Orders Above Customer Average

Return orders whose total_amount is greater than that customer's own average order amount.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q36 — Departments With High Earners

Return departments containing at least one employee earning more than $110,000. Use EXISTS.  
**Tables:** \
**`departments:`** department_id, department_name, location \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q37 — Customers With No Support Tickets

Find customers for whom no support ticket exists. Use NOT EXISTS.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`support_tickets:`** ticket_id, customer_id, assigned_employee_id, created_at, resolved_at, priority, status, subject


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q38 — Project Staffing Pipeline

Use multiple CTEs to calculate headcount and total allocation per project, then return projects with at least 3 assigned employees.  
**Tables:** \
**`projects:`** project_id, project_name, start_date, end_date, status \
**`project_assignments:`** assignment_id, project_id, employee_id, role_on_project, allocation_pct


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q39 — Revenue Share by Category

Use CTEs to calculate sales revenue by product category and each category's percentage of total revenue.  
**Tables:** \
**`products:`** product_id, sku, product_name, category, unit_price, stock_quantity, reorder_level, active \
**`order_items:`** order_item_id, order_id, product_id, quantity, unit_price


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q40 — Repeat Customers

Use a CTE to identify customers with at least 3 orders and return their first_order_date, latest_order_date, and lifetime_order_value.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


## 5. Dates & Time


### Q41 — Orders in Last 30 Days of Dataset

Find orders placed in the 30 days ending on the maximum order_date in the table.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q42 — Monthly Order Trend

Return month, order_count, and total_revenue for each month.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q43 — Month-over-Month Revenue Change

Calculate monthly revenue, previous_month_revenue, absolute_change, and percentage_change.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q44 — Latest Event per User

Return the latest event for every user.  
**Tables:** \
**`events:`** event_id, user_id, event_type, event_ts, session_id, page


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q45 — Daily Active Users

Count distinct active users per event_date.  
**Tables:** \
**`events:`** event_id, user_id, event_type, event_ts, session_id, page


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q46 — Consecutive Login Days

Find user/date pairs where a login occurred on the immediately preceding calendar day for the same user.  
**Tables:** \
**`events:`** event_id, user_id, event_type, event_ts, session_id, page


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q47 — Days to Resolve Tickets

For resolved or closed tickets, calculate the number of days between created_at and resolved_at.  
**Tables:** \
**`support_tickets:`** ticket_id, customer_id, assigned_employee_id, created_at, resolved_at, priority, status, subject


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q48 — Average Resolution Time by Priority

Calculate average ticket resolution time in hours for each priority.  
**Tables:** \
**`support_tickets:`** ticket_id, customer_id, assigned_employee_id, created_at, resolved_at, priority, status, subject


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q49 — Customers Inactive for 60 Days

Relative to the maximum order_date, find customers whose latest order was more than 60 days ago, including customers with no orders.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q50 — Quarterly Revenue

Return year, quarter, order_count, and revenue for completed orders.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


## 6. Deduplication


### Q51 — Duplicate Customer Emails

Identify normalized email addresses that appear more than once in customer_staging.  
**Tables:** \
**`customer_staging:`** staging_id, source_customer_id, first_name, last_name, email, city, updated_at


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q52 — Keep Latest Customer Record

Deduplicate customer_staging by normalized email, keeping the row with the latest updated_at.  
**Tables:** \
**`customer_staging:`** staging_id, source_customer_id, first_name, last_name, email, city, updated_at


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q53 — Duplicate Events

Identify duplicate events based on user_id, event_type, and event_ts.  
**Tables:** \
**`events:`** event_id, user_id, event_type, event_ts, session_id, page


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q54 — Remove Duplicate Events in Result

Return one logical event per user_id, event_type, event_ts using ROW_NUMBER(). Prefer the highest event_id.  
**Tables:** \
**`events:`** event_id, user_id, event_type, event_ts, session_id, page


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q55 — Duplicate Payments

Identify orders with more than one payment row.  
**Tables:** \
**`payments:`** payment_id, order_id, payment_date, amount, payment_method, payment_status


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q56 — Latest Payment per Order

Return only the most recent payment record for each order.  
**Tables:** \
**`payments:`** payment_id, order_id, payment_date, amount, payment_method, payment_status


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q57 — Duplicate Product SKUs

Find duplicated normalized SKUs in product_staging.  
**Tables:** \
**`product_staging:`** staging_id, sku, product_name, category, unit_price, updated_at


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q58 — Canonical Product Staging

Return one row per normalized SKU, keeping the latest updated_at and using staging_id as a tie-breaker.  
**Tables:** \
**`product_staging:`** staging_id, sku, product_name, category, unit_price, updated_at


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q59 — Duplicate Daily Logs

Identify repeated pipeline log records having the same pipeline_name, log_level, message, and logged_at.  
**Tables:** \
**`pipeline_logs:`** log_id, pipeline_name, log_level, message, logged_at


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q60 — Deduplicated Log Feed

Return a deduplicated pipeline log feed, keeping the highest log_id for each duplicate group.  
**Tables:** \
**`pipeline_logs:`** log_id, pipeline_name, log_level, message, logged_at


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


## 7. Top-N Problems


### Q61 — Top 3 Customers by Revenue

Return the top 3 customers by total completed-order revenue.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q62 — Second-Highest Company Salary

Return all employees whose salary equals the second-highest distinct salary in the company.  
**Tables:** \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q63 — Top 3 Salaries per Department

Return the employees in the top 3 distinct salary levels within each department.  
**Tables:** \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q64 — Latest Transaction per Customer

Return each customer's latest transaction from transactions.  
**Tables:** \
**`transactions:`** transaction_id, customer_id, transaction_ts, transaction_type, amount, channel


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q65 — Top 2 Products per Category

Rank products by units sold within category and return the top 2 products per category.  
**Tables:** \
**`products:`** product_id, sku, product_name, category, unit_price, stock_quantity, reorder_level, active \
**`order_items:`** order_item_id, order_id, product_id, quantity, unit_price


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q66 — Highest-Value Order per Month

Return the highest-value order in each month. Return ties.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q67 — Top 3 Cities by Revenue

Return the top 3 customer cities by total order revenue.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q68 — Most Active Users

Return the 5 users with the highest number of events.  
**Tables:** \
**`events:`** event_id, user_id, event_type, event_ts, session_id, page


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q69 — Third-Latest Order per Customer

Return the third-latest order for customers who have at least 3 orders.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q70 — Top Employee by Project Load

Return the employee or employees with the greatest sum of allocation_pct across all project assignments.  
**Tables:** \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title \
**`project_assignments:`** assignment_id, project_id, employee_id, role_on_project, allocation_pct


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


## 8. Real-World Data Problems


### Q71 — Customer 360 Snapshot

Build one result per customer with signup_date, segment, lifetime_orders, lifetime_revenue, latest_order_date, and open_ticket_count.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount \
**`support_tickets:`** ticket_id, customer_id, assigned_employee_id, created_at, resolved_at, priority, status, subject


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q72 — Potentially Stuck Pipelines

Find pipeline runs/log groups with ERROR logs or with a duration greater than 90 minutes. Return pipeline_name, run_date, error_count, and duration_minutes.  
**Tables:** \
**`pipeline_logs:`** log_id, pipeline_name, log_level, message, logged_at


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q73 — Checkout Conversion Funnel

Using events, calculate distinct users who viewed a product, added to cart, and purchased, plus view-to-purchase conversion percentage.  
**Tables:** \
**`events:`** event_id, user_id, event_type, event_ts, session_id, page


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q74 — Revenue Reconciliation

Compare each order's total_amount with the sum of its order_items and return orders where the absolute difference is greater than $0.01.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount \
**`order_items:`** order_item_id, order_id, product_id, quantity, unit_price


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q75 — Customer Churn Candidates

Relative to the dataset's maximum order_date, find customers with at least 2 historical orders but no order in the last 90 days.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q76 — Payment Failure Rate

For each month, calculate total payment attempts, failed payments, and failure_rate_pct.  
**Tables:** \
**`payments:`** payment_id, order_id, payment_date, amount, payment_method, payment_status


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q77 — Support SLA Breaches

Assume SLA targets are High=24h, Medium=48h, Low=72h. Return resolved/closed tickets that exceeded their SLA.  
**Tables:** \
**`support_tickets:`** ticket_id, customer_id, assigned_employee_id, created_at, resolved_at, priority, status, subject


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q78 — Pipeline Error Trend

For each pipeline and day, calculate total logs, error logs, and error_rate_pct.  
**Tables:** \
**`pipeline_logs:`** log_id, pipeline_name, log_level, message, logged_at


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q79 — Product Reorder Candidates

Return active products with stock_quantity below reorder_level, along with units sold in the last 30 days relative to the dataset's max order date.  
**Tables:** \
**`products:`** product_id, sku, product_name, category, unit_price, stock_quantity, reorder_level, active \
**`order_items:`** order_item_id, order_id, product_id, quantity, unit_price \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)


### Q80 — Suspicious Transaction Pattern

Find customers who had at least 3 transactions within the same calendar day and whose daily transaction amount exceeded $1,000.  
**Tables:** \
**`transactions:`** transaction_id, customer_id, transaction_ts, transaction_type, amount, channel


In [ ]:
sql = """
-- Write your SQL answer here


"""

run_sql(sql)
